# Persona steering (EXPERIMENTAL)

These persona variants are experimental. In the j-steer-dev experiments,
persona-contrast pullbacks FAILED specificity controls: they steered
generations, but no more selectively than an unrelated persona's vector did.
(A "pullback" here is `J_l^T @ w`: a direction `w` at the output pulled back to a
residual-stream direction, the standard autodiff name for J-transpose applied to
a cotangent.) Only `word_vector` (see `word_steering.ipynb`) is the verified
method. This notebook exists so you can experiment and compare against a plain
mean_diff baseline, not as a recommendation.

Two variants:

- `persona_vector`: pull back the final-layer activation contrast
  `h_bar(pos) - h_bar(neg)` through the cached Jacobian.
- `persona_topk_vector`: read each persona's mean activation through the
  unembedding, take the top-k tokens it evokes, contrast those tokens'
  unembedding rows, pull that back (persona -> vocabulary bottleneck -> the
  verified word mechanism).

In [ ]:
# demo notebook authored by Claude
import os
import sys
from loguru import logger

logger.remove()  # drop default stderr handler
logger.add(os.sys.stdout, format="<level>{level.icon}</level> {message}")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from jsteer import Jacobian

sys.path.insert(0, "..")  # repo root for config.py
import config
from jlens.examples import load_wikitext_prompts

MODEL = "Qwen/Qwen3-0.6B"
# MODEL = "Qwen/Qwen3.5-4B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to("cuda").eval()

# fit-or-load the cache for THIS model (builds on first run, loads after).
# The lambda means WikiText is only streamed on a cache MISS.
jac = Jacobian.fit_cached(model, tok, lambda: load_wikitext_prompts(128),
                          config.cache_path(MODEL), layers=(0.3, 0.9))
jac

## The persona contrast: optimist vs pessimist

Eight short first-person statements per side. These are the prompts whose
final-layer mean activations get contrasted (and, for the mean_diff baseline,
the pos/neg training prompts).

In [ ]:
optimist = [
    "Things usually work out better than people expect, and today is no exception.",
    "Every setback I have hit this year turned into a door I could not have planned for.",
    "The team is behind schedule, but honestly the hard part is done and the rest is downhill.",
    "I love how much there is to look forward to this month.",
    "Even the rainy days lately have felt like a good excuse to slow down and enjoy the quiet.",
    "The new neighbours seem wonderful, and I think this street keeps getting friendlier.",
    "Whatever happens with the results, we learned so much that we already came out ahead.",
    "I woke up early, the coffee was perfect, and I am certain this week is going to be great.",
]
pessimist = [
    "Things usually go worse than people expect, and today is no exception.",
    "Every setback this year just confirmed that planning is pointless.",
    "The team is behind schedule, and frankly the hardest part has not even started.",
    "I dread how much is crammed into this month.",
    "The rainy days lately just make everything feel heavier and more pointless.",
    "The new neighbours seem like trouble, and this street keeps getting worse.",
    "Whatever happens with the results, it will not make up for the time we wasted.",
    "I woke up tired, the coffee was burnt, and I am certain this week is going to drag.",
]

def gen(vec, prompt, C, do_sample=False, max_new_tokens=40, seed=0):
    enc = tok(prompt, return_tensors="pt").to(model.device)
    torch.manual_seed(seed)
    with vec(model, C=C):
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=do_sample,
                             temperature=0.7 if do_sample else None,
                             top_p=0.95 if do_sample else None,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)

PROMPT = "Here is my honest assessment of how the project is going:"

## persona_vector (EXPERIMENTAL)

Pulls `h_bar(optimist) - h_bar(pessimist)` back through the Jacobian.
SHOULD: +C reads more upbeat than C=0; expect the effect to be blunter and
less specific than the word vector. On this 0.6B model -C does NOT produce
coherent negative tone: it degenerates into repetition (see output below).
The vector moves tone in one direction and breaks the model in the other,
which is itself a datum about how crude the persona contrast is.

In [ ]:
v_persona = jac.persona_vector(model, tok, optimist, pessimist)
for C in (-2, 0, 2):
    print(f"C={C:+d}: {gen(v_persona, PROMPT, C)!r}")

## persona_topk_vector (EXPERIMENTAL)

Same personas through the vocabulary bottleneck. The logged top-k tokens are
worth reading (read your data): they show WHAT each persona's mean activation
actually evokes at the final layer.

On this setup the readout is a null result, and the log makes it legible:
both personas' top-8 are the SAME generic sentence starters (" I", " The",
" So", ...), because the mean next token after a first-person statement is a
new sentence start regardless of valence. Identical token sets means the
contrast is exactly zero, so the vector is null and the generations below do
not move at all. Larger k does not help (tested k=32/64: the extra tokens are
still shared, so the contrast is ordering noise). If you use this variant,
check this log first; steering only makes sense when the two token sets
actually differ.

In [ ]:
v_topk = jac.persona_topk_vector(model, tok, optimist, pessimist, k=8)
for C in (-2, 0, 2):
    print(f"C={C:+d}: {gen(v_topk, PROMPT, C)!r}")

## mean_diff baseline (steering-lite)

The standard activation-difference method on the same prompts and layers, for
comparison. No Jacobian involved: it contrasts mid-layer activations directly.

In [ ]:
from steering_lite import Vector, MeanDiffC

v_md = Vector.train(model, tok, optimist, pessimist, MeanDiffC(layers=tuple(jac.layers)))
for C in (-2, 0, 2):
    print(f"C={C:+d}: {gen(v_md, PROMPT, C)!r}")

## What to take away

On this 0.6B setup: `mean_diff` moves tone coherently in both directions at
C around 2; `persona_vector` moves it upbeat at +2 but degenerates into
repetition at -2 (not coherent negative tone); `persona_topk_vector`
collapses to a null vector because the two personas evoke the same
final-layer vocabulary. Moving tone is not the interesting question, though.
The j-steer-dev specificity controls asked whether a persona vector moves ITS
OWN axis more than an unrelated persona's vector does, and the persona
pullbacks failed that test. If you need targeted steering, use `word_vector`;
treat everything in this notebook as raw material for experiments.